<h3>Importando bibliotecas que vamos utilizar

In [ ]:
import os
import dotenv

import requests

from pymongo import MongoClient, InsertOne
from pymongo.errors import InvalidURI

<h4>Importando variáveis de ambiente

In [ ]:
dotenv.load_dotenv()

<h3> Definindo constantes para requisição

In [ ]:
#URL constante utilizada para a extração das análises
STEAM_API_URL = "https://store.steampowered.com/appreviews"

#ID do jogo na Steam que será extraído as análises. Para obter o ID, basta ir à página do jogo no site da Steam e extrair o código númerico presente na URL do jogo
steam_game_id = 1174180

#Parâmetros da requisição. Pode ser alterada de acordo com sua necessidade. Consulte https://partner.steamgames.com/doc/store/getreviews para mais informações
request_params = {
    "json": 1,
    "filter": "recent",
    "language": "brazilian",
    "num_per_page": 100
}

<h3> Realizando conexão de dados com o banco de dados Mongo

In [ ]:
try:
    #Tenta realizar a conexão com o servidor Mongo. Caso contrário, resultará em um erro
    mongo_client = MongoClient( os.getenv("MONGO_URI") )
    
    #Definindo database e collection utilizadas pelo notebook. Quando utilizamos pymongo, inicialmente o database e collection não são criadas, apenas quando inserimos de fato os dados extraídos
    reviews_database = mongo_client[ os.getenv("MONGO_DATABASE") ]
    reviews_collection = reviews_database[ os.getenv("MONGO_COLLECTION") ]
    
except InvalidURI as err:
    
    #Mensagem de erro quando URI é inválida
    raise Exception("Uma MONGO_URI inválida foi passada! Defina de forma adequada as váriaveis de ambiente." )

<h3> Extraindo análises feitas pelos jogadores

In [ ]:

while True:
    
    #Utilizando a biblioteca requests, requisitamos as reviews com base nos parâmetros utilizados acima
    reviews_response = requests.get( f"{ STEAM_API_URL }/{ steam_game_id }", params = request_params )
    reviews_dict = reviews_response.json()
  
    #Quando todas as análises forem extraidas, a API entrega uma response com nenhuma análise. Dessa forma, podemos utilizar esse fator como parâmetro de parada do loop while
    if reviews_dict["query_summary"]["num_reviews"] == 0:
        
        break
    
    #Cria uma lista de instruções InsertOne para utilizar na escrita bulk na collection Mongo
    reviews_bulk = [ InsertOne( {**review, "steam_game_id": steam_game_id} ) for review in reviews_dict["reviews"]  ]
    reviews_collection.bulk_write( reviews_bulk )
    
    #Cada response trás o valor da próxima página de análises. Modificamos o parâmetro da requisição para a próxima execução do loop trazer análises novas
    request_params["cursor"] = reviews_dict["cursor"]
    

    